# MTMC 11a — WILDTRACK Stages 0-2 (Tracking + ReID Features)
Person multi-camera tracking pipeline on WILDTRACK dataset (7 cameras, 1920x1080, 2fps).

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _match = re.search(r"(\d+)\.(\d+)", _cap)
        if _match:
            _major, _minor = _match.groups()
            _sm = int(_major) * 10 + int(_minor)
            if _sm < 70:
                print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) \u2014 installing compatible PyTorch ...")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])
                print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Clone repository
REPO_URL = "https://github.com/abdoibrahim257/MTC-repo.git"
PROJECT = Path("/kaggle/working/gp")

if not PROJECT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
    print(f"Cloned to {PROJECT}")
else:
    print(f"Repository already exists at {PROJECT}")

os.chdir(str(PROJECT))

# Install dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.1", "boxmot>=10.0", "faiss-cpu>=1.7",
    "timm>=0.9", "omegaconf>=2.3", "networkx>=3.1",
    "opencv-python-headless", "torchreid",
    "scipy", "scikit-learn", "click", "tqdm", "Pillow"])
print("Dependencies installed")

In [ ]:
import shutil, os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

WEIGHTS_DIR = Path("/kaggle/input/mtmc-weights")
if not WEIGHTS_DIR.exists():
    raise FileNotFoundError(f"Weights dataset not found at {WEIGHTS_DIR}")

# Copy model files to project
models_dst = PROJECT / "models"
for subdir in ["reid", "detection", "tracker"]:
    src = WEIGHTS_DIR / "models" / subdir
    dst = models_dst / subdir
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.is_file():
                dst_file = dst / f.name
                if not dst_file.exists():
                    shutil.copy2(str(f), str(dst_file))
                    print(f"  Copied {subdir}/{f.name} ({f.stat().st_size/1024**2:.1f} MB)")
                else:
                    print(f"  Already exists: {subdir}/{f.name}")

# Verify critical person model
person_model = models_dst / "reid" / "person_transreid_vit_base_market1501.pth"
assert person_model.exists(), f"Person ReID model not found at {person_model}"
print(f"\nPerson ReID model: {person_model} ({person_model.stat().st_size/1024**2:.1f} MB)")

In [ ]:
import os, sys, subprocess, cv2
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

# Find WILDTRACK dataset
WILDTRACK_INPUT = None
for candidate in [
    Path("/kaggle/input/large-scale-multicamera-detection-dataset"),
    Path("/kaggle/input/wildtrack"),
]:
    if candidate.exists():
        WILDTRACK_INPUT = candidate
        break

if WILDTRACK_INPUT is None:
    raise FileNotFoundError("WILDTRACK dataset not found in /kaggle/input/")
print(f"WILDTRACK dataset: {WILDTRACK_INPUT}")

# Find Image_subsets (may be nested under Wildtrack_dataset/)
img_subsets = None
for d in sorted(WILDTRACK_INPUT.rglob("Image_subsets")):
    img_subsets = d
    break
if img_subsets is None:
    print("Available paths:")
    for p in sorted(WILDTRACK_INPUT.iterdir()):
        print(f"  {p}")
    raise FileNotFoundError("Image_subsets not found")

print(f"Image subsets: {img_subsets}")
cameras = sorted([d.name for d in img_subsets.iterdir() if d.is_dir()])
print(f"Cameras: {cameras}")

# Set up WILDTRACK directory
WILDTRACK_DIR = PROJECT / "data" / "raw" / "wildtrack"
VIDEOS_DIR = WILDTRACK_DIR / "videos"
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Generate MP4 videos from frames
for cam in cameras:
    cam_dir = img_subsets / cam
    frames = sorted(cam_dir.glob("*.png"))
    if not frames:
        frames = sorted(cam_dir.glob("*.jpg"))
    if not frames:
        print(f"  {cam}: No frames found, skipping")
        continue

    out_path = VIDEOS_DIR / f"{cam}.mp4"
    if out_path.exists():
        print(f"  {cam}: Video exists ({out_path.stat().st_size/1024**2:.1f} MB)")
        continue

    first = cv2.imread(str(frames[0]))
    h, w = first.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, 2.0, (w, h))
    for i, fp in enumerate(frames):
        writer.write(cv2.imread(str(fp)))
        if (i + 1) % 100 == 0:
            print(f"  {cam}: {i+1}/{len(frames)}")
    writer.release()
    print(f"  {cam}: {len(frames)} frames -> {out_path.name} ({w}x{h}, {out_path.stat().st_size/1024**2:.1f} MB)")

# Symlink annotations and calibrations
ann_src = cal_src = None
for d in WILDTRACK_INPUT.rglob("annotations_positions"):
    ann_src = d
    break
for d in WILDTRACK_INPUT.rglob("calibrations"):
    cal_src = d
    break

for name, src in [("annotations_positions", ann_src), ("calibrations", cal_src)]:
    dst = WILDTRACK_DIR / name
    if src and not dst.exists():
        dst.symlink_to(src)
        print(f"Linked {name}")
    elif src:
        print(f"Already linked: {name}")
    else:
        print(f"WARNING: {name} not found!")

print(f"\nVideos: {list(VIDEOS_DIR.glob('*.mp4'))}")

In [ ]:
import subprocess, sys, os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

result = subprocess.run(
    [sys.executable, "scripts/prepare_dataset.py", "--dataset", "wildtrack",
     "--root", "data/raw/wildtrack"],
    capture_output=True, text=True, cwd=str(PROJECT)
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"GT preparation failed (exit code {result.returncode})")

gt_dir = PROJECT / "data" / "raw" / "wildtrack" / "manifests" / "ground_truth"
if gt_dir.exists():
    for f in sorted(gt_dir.glob("*")):
        if f.is_file():
            n = len(f.read_text().strip().split("\n"))
            print(f"  {f.name}: {n} lines")
else:
    print(f"WARNING: GT not found at {gt_dir}")

In [ ]:
import subprocess, sys, os, time, json
from pathlib import Path
from datetime import datetime

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"wildtrack_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Run: {RUN_NAME}")

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/wildtrack.yaml",
    "--stages", "0,1,2",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", "stage1.detector.confidence_threshold=0.55",
    "--override", "stage1.tracker.min_hits=3",
    "--override", "stage1.tracker.match_thresh=0.75",
    "--override", "stage1.min_tracklet_length=8",
    "--override", "stage2.pca.n_components=256",
    "--override", "stage2.camera_bn.enabled=true",
    "--override", "stage2.reid.flip_augment=true",
]

print(f"CMD: {' '.join(cmd)}")
t0 = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT))
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min (exit={result.returncode})")

run_dir = DATA_OUT / RUN_NAME
for stage in ["stage0", "stage1", "stage2"]:
    sd = run_dir / stage
    if sd.exists():
        nf = sum(1 for _ in sd.rglob("*") if _.is_file())
        sz = sum(f.stat().st_size for f in sd.rglob("*") if f.is_file()) / 1024**2
        print(f"  {stage}: {nf} files ({sz:.1f} MB)")
    else:
        print(f"  {stage}: MISSING")

In [ ]:
import tarfile, json, os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
DATA_OUT = Path("/tmp/pipeline_outputs")
OUTPUT = Path("/kaggle/working")

runs = sorted(DATA_OUT.glob("wildtrack_*"))
if not runs:
    raise FileNotFoundError("No wildtrack run directory found")
run_dir = runs[-1]
RUN_NAME = run_dir.name
print(f"Packaging: {RUN_NAME}")

metadata = {"run_name": RUN_NAME, "dataset": "wildtrack", "type": "person"}
meta_path = DATA_OUT / "run_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

gt_dir = PROJECT / "data" / "raw" / "wildtrack" / "manifests" / "ground_truth"
ckpt_path = OUTPUT / "checkpoint.tar.gz"

with tarfile.open(str(ckpt_path), "w:gz") as tar:
    tar.add(str(meta_path), arcname="run_metadata.json")
    for stage in ["stage1", "stage2"]:
        sd = run_dir / stage
        if sd.exists():
            tar.add(str(sd), arcname=f"{RUN_NAME}/{stage}")
            print(f"  Added {stage}")
    manifest = run_dir / "stage0" / "frames_manifest.json"
    if manifest.exists():
        tar.add(str(manifest), arcname=f"{RUN_NAME}/stage0/frames_manifest.json")
        print("  Added stage0/frames_manifest.json")
    if gt_dir.exists():
        tar.add(str(gt_dir), arcname="gt_annotations")
        print("  Added gt_annotations")

sz = ckpt_path.stat().st_size / 1024**2
print(f"\nCheckpoint: {ckpt_path} ({sz:.1f} MB)")
summary = {"run_name": RUN_NAME, "dataset": "wildtrack", "stages": [0,1,2], "size_mb": round(sz,1)}
with open(OUTPUT / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))